[Reference](https://pub.towardsai.net/are-tabular-foundation-models-ready-to-replace-gradient-boosting-models-cb039b955162$0)

In [1]:
from xgboost import XGBClassifier

clf = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05)
clf.fit(X_train, y_train)      # builds 300 trees sequentially
preds = clf.predict(X_test)


from tabpfn import TabPFNClassifier

clf = TabPFNClassifier()
clf.fit(X_train, y_train)      # caches the training data as context
preds = clf.predict(X_test)

In [3]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.5 MB/s eta 0:00:00


In [4]:
import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

def benchmark(name, model):
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    fit_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    predict_time = time.perf_counter() - t0

    print(f"{name:10s} fit={fit_time:.4f}s  predict={predict_time:.4f}s  "
          f"acc={accuracy_score(y_test, preds):.4f}  "
          f"auc={roc_auc_score(y_test, proba):.4f}")

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

benchmark("XGBoost", XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    eval_metric="logloss", random_state=42))

benchmark("CatBoost", CatBoostClassifier(
    iterations=300, depth=6, learning_rate=0.05,
    verbose=False, random_state=42))

XGBoost    fit=0.5606s  predict=0.0039s  acc=0.9474  auc=0.9931
CatBoost   fit=7.3831s  predict=0.0146s  acc=0.9561  auc=0.9960


In [6]:
!pip install tabpfn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 732.3/732.3 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.0 MB/s eta 0:00:00


In [8]:
from tabpfn import TabPFNClassifier

benchmark("TabPFN", TabPFNClassifier())

In [10]:
!pip install shap

In [13]:
import shap
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

xgb_model = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    eval_metric="logloss", random_state=42,
)
xgb_model.fit(X_train, y_train)

cat_model = CatBoostClassifier(
    iterations=300, depth=6, learning_rate=0.05,
    verbose=False, random_state=42,
)
cat_model.fit(X_train, y_train)

# TreeExplainer works for both XGBoost and CatBoost out of the box
xgb_shap_values = shap.TreeExplainer(xgb_model).shap_values(X_test)
cat_shap_values = shap.TreeExplainer(cat_model).shap_values(X_test)

xgb_top = feature_names[np.argsort(-np.abs(xgb_shap_values).mean(axis=0))[0]]
cat_top = feature_names[np.argsort(-np.abs(cat_shap_values).mean(axis=0))[0]]
print("Top feature by mean |SHAP| (XGBoost):", xgb_top)
print("Top feature by mean |SHAP| (CatBoost):", cat_top)

Top feature by mean |SHAP| (XGBoost): worst perimeter
Top feature by mean |SHAP| (CatBoost): worst area


In [14]:
import numpy as np
import tabpfn
import shapiq

clf = tabpfn.TabPFNClassifier()
clf.fit(X_train, y_train)


explainer = shapiq.TabPFNExplainer(
    model=clf,
    data=X_train,
    labels=y_train,
    index="SV",
    max_order=1,

)
n_explain = 5
budget = 16
values = np.zeros((n_explain, X_test.shape[1]))
for i in range(n_explain):
    iv = explainer.explain(X_test[i : i + 1], budget=budget, random_state=0)
    for feature_idx, val in iv.dict_values.items():
        if len(feature_idx) == 1:
            values[i, feature_idx[0]] = val

mean_abs_shap = np.abs(values).mean(axis=0)
top_feature = feature_names[np.argsort(-mean_abs_shap)[0]]
print("Top feature by mean |SHAP| (TabPFN):", top_feature)